# Distributed RL Training Architecture

## Overview

This notebook explores the distributed training infrastructure of the RL Platform,
including parallel rollout workers, parameter servers, and asynchronous training pipelines.

---

## Architecture: Parameter Server Pattern

```mermaid
graph TB
    subgraph Central Learner
        PS[Parameter Server]
        GU[Gradient Updater]
        RB[Replay Buffer]
    end
    
    subgraph Rollout Workers
        W1[Worker 0]
        W2[Worker 1]
        W3[Worker 2]
        W4[Worker N]
    end

    PS -->|broadcast weights| W1
    PS -->|broadcast weights| W2
    PS -->|broadcast weights| W3
    PS -->|broadcast weights| W4
    
    W1 -->|experience| RB
    W2 -->|experience| RB
    W3 -->|experience| RB
    W4 -->|experience| RB
    
    RB -->|sample batch| GU
    GU -->|update params| PS
```

## Throughput Analysis

| Workers | Episodes/sec | Speedup | Efficiency |
|---------|-------------|---------|------------|
| 1       | 45          | 1.0×    | 100%       |
| 2       | 86          | 1.9×    | 96%        |
| 4       | 162         | 3.6×    | 90%        |
| 8       | 296         | 6.6×    | 82%        |

Efficiency decreases with more workers due to communication overhead and synchronization costs.

In [ ]:
import sys; sys.path.insert(0, '..')
import matplotlib.pyplot as plt
import numpy as np

# Simulated throughput scaling
workers = [1, 2, 4, 8, 16]
throughput = [45, 86, 162, 296, 510]
efficiency = [t / (workers[i] * throughput[0]) for i, t in enumerate(throughput)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(workers, throughput, 'o-', color='steelblue', linewidth=2, markersize=8)
ax1.plot(workers, [45*w for w in workers], '--', color='gray', alpha=0.6, label='Linear ideal')
ax1.set_xlabel('Number of Workers'); ax1.set_ylabel('Episodes / second')
ax1.set_title('Distributed Training Throughput', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3); ax1.set_xscale('log', base=2)

ax2.bar([str(w) for w in workers], [e*100 for e in efficiency], color='darkorange', alpha=0.8)
ax2.set_xlabel('Number of Workers'); ax2.set_ylabel('Parallel Efficiency (%)')
ax2.set_title('Worker Efficiency', fontweight='bold')
ax2.axhline(y=80, color='red', linestyle='--', alpha=0.6, label='80% threshold')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Distributed Training Configuration

```mermaid
sequenceDiagram
    participant L as Central Learner
    participant W1 as Worker 1
    participant W2 as Worker 2
    participant Q as Experience Queue
    
    L->>W1: broadcast_weights(θ_t)
    L->>W2: broadcast_weights(θ_t)
    
    W1->>Q: push(trajectories_1)
    W2->>Q: push(trajectories_2)
    
    Q->>L: sample_batch(B)
    L->>L: gradient_update(batch)
    Note over L: θ_{t+1} = θ_t - α∇L
    
    L->>W1: broadcast_weights(θ_{t+1})
    L->>W2: broadcast_weights(θ_{t+1})
```

In [ ]:
from src.training.distributed_trainer import DistributedConfig
from src.envs.graph_nav_env import EnvConfig

dist_config = DistributedConfig(
    num_workers=4,
    episodes_per_worker=8,
    num_iterations=10,  # Short demo
    embed_dim=128,
)
env_config = EnvConfig(grid_size=4, coin_nodes={3, 7, 12})

print(f'Distributed config:')
print(f'  Workers: {dist_config.num_workers}')
print(f'  Episodes/worker: {dist_config.episodes_per_worker}')
print(f'  Total ep/iter: {dist_config.num_workers * dist_config.episodes_per_worker}')

## Replay Buffer Architecture

```mermaid
graph LR
    subgraph Workers
        W1[Worker 1] --> |push| RB
        W2[Worker 2] --> |push| RB
        W3[Worker 3] --> |push| RB
    end
    
    subgraph Replay Buffer
        RB[Ring Buffer\n100K capacity]
        ST[Sum Tree\nPER priorities]
    end
    
    RB --> |sample B=256| L[Central Learner]
    L --> |update priorities| ST
    ST --> |priority weights| RB
```

### Prioritized Experience Replay (PER)

PER samples transitions with probability:
$$P(i) = \frac{p_i^\alpha}{\sum_k p_k^\alpha}$$

where $p_i = |\delta_i| + \epsilon$ is the absolute TD error.

Importance sampling corrects for the non-uniform sampling:
$$w_i = \left(\frac{1}{N \cdot P(i)}\right)^\beta$$